# 02 — Signal Preprocessing, AASM Harmonization & Artifact QC

**Purpose:** Convert raw Sleep-EDF PSG recordings into stable, normalized 30-second epochs for the final model.

`Raw PSG → channel selection → 0.5–35 Hz bandpass → 50 Hz notch → 30 s epochs → QC flags → z-score normalization → cached subject-night arrays`

### Final-pipeline rule
The preprocessing contract is fixed:
- 4 channels: EEG Fpz-Cz, EEG Pz-Oz, EOG horizontal, EMG submental
- 0.5–35 Hz fourth-order Butterworth bandpass
- 50 Hz IIR notch
- AASM five-stage harmonization
- per-channel, per-epoch z-score normalization
- artifact QC flags retained explicitly


In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import mne
from scipy.signal import butter, sosfiltfilt, iirnotch, tf2sos

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

MANIFEST_PATH = Path("/home/shamique/projects/sleep/data/manifests/sleep_edf.csv")
CACHE_DIR = Path("/home/shamique/projects/sleep/data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EPOCH_SEC = 30
CHANNELS = {
    "eeg1": "EEG Fpz-Cz",
    "eeg2": "EEG Pz-Oz",
    "eog": "EOG horizontal",
    "emg": "EMG submental",
}
BANDPASS = (0.5, 35.0)
NOTCH_FREQ = 50.0

AASM_MAP = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
    "Sleep stage ?": None,
    "Movement time": None,
}
STAGE_NAMES = ["Wake", "N1", "N2", "N3", "REM"]

manifest = pd.read_csv(MANIFEST_PATH)
display(manifest.head())


,subject_id,night,psg,hypnogram,split
0,SC4001,0,/home/shamique/projects/sleep/data/raw/sleep_e...,/home/shamique/projects/sleep/data/raw/sleep_e...,train
1,SC4002,0,/home/shamique/projects/sleep/data/raw/sleep_e...,/home/shamique/projects/sleep/data/raw/sleep_e...,train
2,SC4011,0,/home/shamique/projects/sleep/data/raw/sleep_e...,/home/shamique/projects/sleep/data/raw/sleep_e...,train
3,SC4012,0,/home/shamique/projects/sleep/data/raw/sleep_e...,/home/shamique/projects/sleep/data/raw/sleep_e...,train


## 1. Deterministic filtering utilities


In [2]:
def bandpass_sos(low, high, fs, order=4):
    return butter(order, [low, high], btype="bandpass", fs=fs, output="sos")

def notch_sos(freq, fs, q=30.0):
    b, a = iirnotch(freq, q, fs)
    return tf2sos(b, a)

def filter_signal(x, fs):
    x = sosfiltfilt(bandpass_sos(*BANDPASS, fs=fs), x, axis=-1)
    x = sosfiltfilt(notch_sos(NOTCH_FREQ, fs=fs), x, axis=-1)
    return x


## 2. Load one recording and harmonize annotations to five stages


In [3]:
def load_recording(psg_path, hyp_path):
    raw = mne.io.read_raw_edf(psg_path, preload=True)
    fs = float(raw.info["sfreq"])

    missing = [name for name in CHANNELS.values() if name not in raw.ch_names]
    if missing:
        raise ValueError(f"Missing required channels: {missing}")

    raw.pick_channels(list(CHANNELS.values()))
    raw.set_annotations(mne.read_annotations(hyp_path), emit_warning=False)

    events, _ = mne.events_from_annotations(
        raw,
        event_id=lambda label: AASM_MAP.get(label, None),
        chunk_duration=EPOCH_SEC,
    )

    valid_codes = np.array([0, 1, 2, 3, 4])
    keep = np.isin(events[:, 2], valid_codes)
    events = events[keep]

    epochs = mne.Epochs(
        raw,
        events,
        event_id=None,
        tmin=0,
        tmax=EPOCH_SEC - 1.0 / fs,
        baseline=None,
        preload=True,
        on_missing="ignore",
        verbose=False,
    )

    data = epochs.get_data()
    labels = events[: len(data), 2].astype(np.int64)

    return data, labels, fs


## 3. Artifact QC

QC is intentionally explicit. Epochs are flagged rather than silently changed, so downstream analysis can report the artifact burden.


In [4]:
AMPLITUDE_CLIP_UV = 500.0
FLATLINE_STD_UV = 0.5

def qc_flags(epoch_data_uv):
    flags = {
        "clipped": bool(np.any(np.abs(epoch_data_uv) > AMPLITUDE_CLIP_UV)),
        "flatline": bool(np.any(epoch_data_uv.std(axis=-1) < FLATLINE_STD_UV)),
        "nan_or_inf": bool(~np.isfinite(epoch_data_uv).all()),
    }
    flags["any_flag"] = any(flags.values())
    return flags


## 4. Per-channel z-score normalization


In [5]:
def normalize_epoch(epoch):
    mean = epoch.mean(axis=-1, keepdims=True)
    std = epoch.std(axis=-1, keepdims=True)
    std = np.where(std < 1e-8, 1e-8, std)
    return (epoch - mean) / std


## 5. Process the complete manifest

For each subject-night, a contiguous `.npz` cache is created. This preserves sequence ordering for the 10-epoch model context.


In [6]:
def process_manifest(manifest_df, limit=None):
    records = []

    for idx, row in enumerate(manifest_df.itertuples(index=False)):
        if limit is not None and idx >= limit:
            break

        try:
            data, labels, fs = load_recording(row.psg, row.hypnogram)
            data_uv = data * 1e6

            flags = [qc_flags(epoch) for epoch in data_uv]
            qc_flag = np.array([f["any_flag"] for f in flags], dtype=bool)

            filtered = np.stack(
                [filter_signal(epoch, fs) for epoch in data_uv],
                axis=0,
            )
            normalized = np.stack(
                [normalize_epoch(epoch) for epoch in filtered],
                axis=0,
            )

            out_path = CACHE_DIR / f"{row.subject_id}_night{row.night}.npz"
            np.savez_compressed(
                out_path,
                epochs=normalized.astype(np.float32),
                labels=labels,
                qc_flag=qc_flag,
                fs=np.float32(fs),
                subject_id=row.subject_id,
                night=np.int64(row.night),
                split=row.split,
            )

            records.append({
                "subject_id": row.subject_id,
                "night": row.night,
                "split": row.split,
                "n_epochs": len(labels),
                "n_flagged": int(qc_flag.sum()),
                "cache_path": str(out_path),
            })

            print(
                f"{row.subject_id} night {row.night}: "
                f"{len(labels)} epochs | flagged={int(qc_flag.sum())}"
            )

        except Exception as exc:
            print(f"[SKIP] {row.psg}: {exc}")

    return pd.DataFrame(records)


## 6. Final-cache execution

Run `limit=None` for the exhibition dataset. A smaller limit is acceptable only for smoke-testing the notebook code.


In [7]:
cache_index = process_manifest(manifest, limit=None)
cache_index_path = CACHE_DIR / "cache_index.csv"
cache_index.to_csv(cache_index_path, index=False)

print("Cached recordings:", len(cache_index))
print("Cached epochs:", int(cache_index["n_epochs"].sum()))
print("QC-flagged epochs:", int(cache_index["n_flagged"].sum()))
print("Saved:", cache_index_path)


SC4001 night 0: 2650 epochs | flagged=2580
SC4002 night 0: 2829 epochs | flagged=2424
SC4011 night 0: 2802 epochs | flagged=2705
SC4012 night 0: 2848 epochs | flagged=2637
Cached recordings: 4
Cached epochs: 11129
QC-flagged epochs: 10346
Saved: /home/shamique/projects/sleep/data/cache/cache_index.csv


## 7. Integrity checks


In [8]:
for _, row in cache_index.iterrows():
    d = np.load(row["cache_path"])
    assert np.isfinite(d["epochs"]).all()
    assert set(np.unique(d["labels"])).issubset(set(range(5)))
    assert len(d["labels"]) == len(d["epochs"])
    assert len(d["qc_flag"]) == len(d["labels"])

total_epochs = int(cache_index["n_epochs"].sum())
total_flagged = int(cache_index["n_flagged"].sum())

print(f"Total cached epochs: {total_epochs:,}")
print(f"QC flag rate: {total_flagged / max(total_epochs, 1):.2%}")
print("Preprocessing integrity checks passed.")


Total cached epochs: 11,129
QC flag rate: 92.96%
Preprocessing integrity checks passed.


## Handoff to Notebook 03

Notebook 03 consumes only:

```text
data/cache/cache_index.csv
data/cache/*.npz
```

No raw-signal reprocessing is needed for EDA or model training.
